# Dreambooth

In [1]:
!git clone https://github.com/huggingface/diffusers
!cd diffusers && pip install .
!pip install xformers bitsandbytes

Cloning into 'diffusers'...
remote: Enumerating objects: 116381, done.
remote: Counting objects: 100% (705/705), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 116381 (delta 646), reused 537 (delta 535), pack-reused 115676 (from 2)
Receiving objects: 100% (116381/116381), 88.88 MiB | 29.10 MiB/s, done.
Resolving deltas: 100% (87006/87006), done.
Processing /content/diffusers
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for diffusers: filename=diffusers-0.37.0.dev0-py3-none-any.whl size=4827065 sha256=6e87b1d11e8747ba11a29c5a50369c08f8099b18a696c635a4d44be8636ac029
  Stored in directory: /tmp/pip-ephem-wheel-cache-wozjzzgy/wheels/8a/fc/09/385efb77b455b2fd4a656c950079c93147e1f50ae614e51beb
Successfully built diffusers
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.36.0
    Uninstalling diffusers-0.36.0:
      Successfully uninst

In [3]:
!unzip /content/style_images_dataset.zip

Archive:  /content/style_images_dataset.zip
   creating: style_images_dataset/
  inflating: style_images_dataset/.DS_Store  
  inflating: __MACOSX/style_images_dataset/._.DS_Store  
   creating: style_images_dataset/images/
  inflating: style_images_dataset/images/metadata.jsonl  
  inflating: __MACOSX/style_images_dataset/images/._metadata.jsonl  
  inflating: style_images_dataset/images/img_7.jpeg  
  inflating: __MACOSX/style_images_dataset/images/._img_7.jpeg  
  inflating: style_images_dataset/images/img_6.jpeg  
  inflating: __MACOSX/style_images_dataset/images/._img_6.jpeg  
  inflating: style_images_dataset/images/img_1.jpeg  
  inflating: __MACOSX/style_images_dataset/images/._img_1.jpeg  
  inflating: style_images_dataset/images/img_10.jpeg  
  inflating: __MACOSX/style_images_dataset/images/._img_10.jpeg  
  inflating: style_images_dataset/images/img_11.jpeg  
  inflating: __MACOSX/style_images_dataset/images/._img_11.jpeg  
  inflating: style_images_dataset/images/img_0.jpe

In [4]:
from datasets import load_dataset
dataset = load_dataset("imagefolder", data_dir="/content/style_images_dataset/images")

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
dataset['train'][0]

{'image': <PIL.Image.Image image mode=RGB size=1266x823>,
 'text': 'A portrait of a male with long hair smoking a cigarette in sks style, pale colors.'}

In [ ]:
os.environ["MODEL_NAME"] = "stable-diffusion-v1-5/stable-diffusion-v1-5"
os.environ["INSTANCE_DIR"] = "/content/style_images_dataset/images"
os.environ["OUTPUT_DIR"] = "/content/model_output"

!accelerate launch /content/diffusers/examples/dreambooth/train_dreambooth.py \
  --use_8bit_adam \
  --gradient_checkpointing \
  --enable_xformers_memory_efficient_attention \
  --set_grads_to_none \
  --caption_column="text" \
  --pretrained_model_name_or_path=$MODEL_NAME  \
  --instance_data_dir=$INSTANCE_DIR \
  --instance_prompt="A painting in sks style"
  --output_dir=$OUTPUT_DIR \
  --validation_prompt="a painting of a house, sks style." \
  --num_validation_images=4 \
  --validation_steps=100 \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-6 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=400

In [5]:
import os

os.environ["MODEL_NAME"] = "stable-diffusion-v1-5/stable-diffusion-v1-5"
os.environ["INSTANCE_DIR"] = "/content/style_images_dataset/images"
os.environ["OUTPUT_DIR"] = "/content/model_output_lora"

!accelerate launch /content/diffusers/examples/advanced_diffusion_training/train_dreambooth_lora_sd15_advanced.py \
  --use_8bit_adam \
  --gradient_checkpointing \
  --enable_xformers_memory_efficient_attention \
  --caption_column="text" \
  --pretrained_model_name_or_path=$MODEL_NAME  \
  --dataset_name=$INSTANCE_DIR \
  --output_dir=$OUTPUT_DIR \
  --instance_prompt="" \
  --validation_prompt="a painting of a house, sks style." \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=1 \
  --learning_rate=5e-6 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=400

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-01-08 19:39:08.960253: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767901148.989223   30144 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767901148.999576   30144 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767901149.031712   30144 computation_plac

In [6]:
from diffusers import DiffusionPipeline, UNet2DConditionModel
from transformers import CLIPTextModel
import torch

unet = UNet2DConditionModel.from_pretrained("/content/model_output_lora/unet")

# if you have trained with `--args.train_text_encoder` make sure to also load the text encoder
text_encoder = CLIPTextModel.from_pretrained("/content/model_output_lora/text_encoder")

pipeline = DiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", unet=unet, text_encoder=text_encoder, dtype=torch.float16,
).to("cuda")

prompts = ["A close up portrait of a female in sks style", "Natural landscape painting in sks style", "An abstract painting with a human silhouette in sks style"]
for n, prompt in enumerate(prompts):
    image = pipeline(prompt, num_inference_steps=50, guidance_scale=7.5, generator=torch.manual_seed(0)).images[0]
    image.save(f"sks_style_image_dreambooth_{n}.png")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


OSError: We couldn't connect to 'https://huggingface.co' to load this model, couldn't find it in the cached files and it looks like /content/model_output_lora/unet is not the path to a directory containing a config.json file.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/diffusers/installation#offline-mode'.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [22]:
import torch
torch.cuda.empty_cache()

In [14]:
pipeline = DiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
).to("cuda")

for n, prompt in enumerate(prompts):
    image = pipeline(prompt, num_inference_steps=50, guidance_scale=7.5, generator=torch.manual_seed(0)).images[0]
    image.save(f"sks_style_image_{n}.png")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

In [23]:
del pipeline